# Job Market Demand Forecasting — Multi-Country LSTM / GRU / BiLSTM

**Objective:** Forecast monthly job posting demand per sector using recurrent neural networks trained on the Indeed Job Postings Index across 6 countries (US, AU, CA, DE, FR, GB).

**Dataset:** [Indeed Job Postings Index](https://www.kaggle.com/datasets/kimminh21/job-postings) — Daily seasonally-adjusted index, baseline = 100 (Feb 1, 2020).

**Approach:** Pool data from 6 countries (cross-learning) to increase training data ~6x, then compare LSTM, GRU, and BiLSTM architectures.

---

### Notebook Structure
1. Environment Setup & Imports
2. Multi-Country Dataset Loading
3. Exploratory Data Analysis (EDA)
4. Monthly Aggregation
5. Data Normalization & Cross-Country Pooling
6. Model Training (LSTM vs GRU vs BiLSTM)
7. Evaluation & Comparison
8. Future Forecasting
9. Interpretation & Limitations

## 1. Environment Setup & Imports

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.figsize': (14, 6),
    'figure.dpi': 100,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'lines.linewidth': 2,
    'font.size': 11
})

SEED = 42
np.random.seed(SEED)

import tensorflow as tf
tf.random.set_seed(SEED)
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

print(f"TensorFlow version: {tf.__version__}")
print(f"NumPy version:      {np.__version__}")
print(f"Pandas version:     {pd.__version__}")
print("All imports successful")

## 2. Multi-Country Dataset Loading

The dataset contains sector-level data for 6 countries. We load all of them, find common sectors, and select our targets.

**Kaggle path:** Add the "Indeed Job Postings Index" dataset by Kim Minh via the sidebar.

In [ ]:
INPUT_DIR = '/kaggle/input/datasets/kimminh21/job-postings'

print("Files in dataset:")
print("=" * 60)
for dirname, _, filenames in os.walk(INPUT_DIR):
    level = dirname.replace(INPUT_DIR, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(dirname)}/")
    subindent = ' ' * 2 * (level + 1)
    for f in sorted(filenames):
        filepath = os.path.join(dirname, f)
        size_mb = os.path.getsize(filepath) / (1024 * 1024)
        print(f"{subindent}{f} ({size_mb:.2f} MB)")

In [ ]:
COUNTRIES = {
    'US': 'job_postings_by_sector_US.csv',
    'AU': 'job_postings_by_sector_AU.csv',
    'CA': 'job_postings_by_sector_CA.csv',
    'DE': 'job_postings_by_sector_DE.csv',
    'FR': 'job_postings_by_sector_FR.csv',
    'GB': 'job_postings_by_sector_GB.csv',
}

dfs_raw = {}
for country, filename in COUNTRIES.items():
    filepath = os.path.join(INPUT_DIR, country, filename)
    if os.path.exists(filepath):
        df_c = pd.read_csv(filepath, parse_dates=['date'])
        df_c['country'] = country
        dfs_raw[country] = df_c
        print(f"{country}: {df_c.shape[0]:>7,} rows | "
              f"{df_c['display_name'].nunique():>2} sectors | "
              f"{df_c['date'].min().date()} to {df_c['date'].max().date()}")
    else:
        print(f"{country}: file not found at {filepath}")

print(f"\nLoaded {len(dfs_raw)} countries")

In [ ]:
print("Variable types per country:")
for country, df_c in dfs_raw.items():
    print(f"  {country}: {df_c['variable'].unique()}")

print("\nSector names per country:")
print("=" * 60)
for country, df_c in dfs_raw.items():
    sectors = sorted(df_c['display_name'].unique())
    print(f"\n{country} ({len(sectors)} sectors):")
    for s in sectors:
        print(f"  - {s}")

In [ ]:
dfs_filtered = {}
for country, df_c in dfs_raw.items():
    if 'total postings' in df_c['variable'].values:
        df_f = df_c[df_c['variable'] == 'total postings'].copy()
    elif 'total' in df_c['variable'].values:
        df_f = df_c[df_c['variable'] == 'total'].copy()
    else:
        print(f"{country}: no 'total postings' or 'total' variable found, skipping")
        continue
    dfs_filtered[country] = df_f
    print(f"{country}: {df_f.shape[0]:>7,} rows after filter")

sector_sets = [set(df_f['display_name'].unique()) for df_f in dfs_filtered.values()]
common_sectors = sector_sets[0]
for s in sector_sets[1:]:
    common_sectors = common_sectors.intersection(s)

print(f"\nCommon sectors across all {len(dfs_filtered)} countries: {len(common_sectors)}")
for s in sorted(common_sectors):
    print(f"  - {s}")

In [ ]:
from collections import Counter

DESIRED_SECTORS = [
    'Software Development',
    'Data & Analytics',
    'IT Systems & Solutions',
    'Project Management',
    'Marketing',
    'Management',
    'Banking & Finance',
    'Human Resources',
]

MIN_COUNTRIES = 4

all_sectors_flat = [s for df_f in dfs_filtered.values() for s in df_f['display_name'].unique()]
sector_counts = Counter(all_sectors_flat)

TARGET_SECTORS = [s for s in DESIRED_SECTORS if sector_counts.get(s, 0) >= MIN_COUNTRIES]
not_included = [s for s in DESIRED_SECTORS if s not in TARGET_SECTORS]

print(f"Target sectors (available in >= {MIN_COUNTRIES} countries): {len(TARGET_SECTORS)}/{len(DESIRED_SECTORS)}\n")
for s in TARGET_SECTORS:
    countries_with = [c for c, df_f in dfs_filtered.items() if s in df_f['display_name'].values]
    in_all = "ALL" if len(countries_with) == len(dfs_filtered) else f"{len(countries_with)}/{len(dfs_filtered)}"
    print(f"  {s:25s} [{in_all}] ({', '.join(countries_with)})")

if not_included:
    print(f"\nExcluded (fewer than {MIN_COUNTRIES} countries):")
    for s in not_included:
        present_in = [c for c, df_f in dfs_filtered.items() if s in df_f['display_name'].values]
        print(f"  {s} (only in: {', '.join(present_in) if present_in else 'none'})")

partial = [s for s in TARGET_SECTORS if s not in common_sectors]
if partial:
    print(f"\n{len(partial)} sector(s) not in all countries — pooling will use available subset:")
    for s in partial:
        missing_from = [c for c, df_f in dfs_filtered.items() if s not in df_f['display_name'].values]
        print(f"  {s}: missing from {', '.join(missing_from)}")

## 3. Exploratory Data Analysis (EDA)

We explore the temporal structure across countries to validate the cross-learning approach:
- Do all countries follow a similar COVID crash-recovery pattern?
- Are sector trajectories comparable across geographies?
- Any data quality issues?

In [ ]:
sample_sector = TARGET_SECTORS[0]

fig, ax = plt.subplots(figsize=(16, 8))
colors = plt.cm.Set2(np.linspace(0, 1, len(dfs_filtered)))

for (country, df_f), color in zip(dfs_filtered.items(), colors):
    mask = df_f['display_name'] == sample_sector
    if mask.sum() == 0:
        continue
    sector_data = df_f[mask].sort_values('date')
    ax.plot(sector_data['date'], sector_data['indeed_job_postings_index'],
            label=country, alpha=0.85, color=color)

ax.axhline(y=100, color='black', linestyle='--', alpha=0.4, label='Baseline (100)')
ax.axvspan(pd.Timestamp('2020-03-01'), pd.Timestamp('2020-06-01'),
           alpha=0.1, color='red', label='COVID shock')

ax.set_title(f'{sample_sector} — Cross-Country Comparison (Daily)', fontsize=16)
ax.set_xlabel('Date')
ax.set_ylabel('Job Postings Index (100 = Feb 2020)')
ax.legend(loc='upper left', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('cross_country_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
df_us = dfs_filtered['US']
df_us_sectors = df_us[df_us['display_name'].isin(TARGET_SECTORS)].copy()

fig, ax = plt.subplots(figsize=(16, 8))

for sector in TARGET_SECTORS:
    mask = df_us_sectors['display_name'] == sector
    sector_data = df_us_sectors[mask].sort_values('date')
    ax.plot(sector_data['date'], sector_data['indeed_job_postings_index'],
            label=sector, alpha=0.85)

ax.axhline(y=100, color='black', linestyle='--', alpha=0.4, label='Pre-pandemic baseline (100)')
ax.axvspan(pd.Timestamp('2020-03-01'), pd.Timestamp('2020-06-01'),
           alpha=0.1, color='red', label='COVID shock period')

ax.set_title('Indeed Job Postings Index — US Market (Daily)', fontsize=16)
ax.set_xlabel('Date')
ax.set_ylabel('Job Postings Index (100 = Feb 2020)')
ax.legend(loc='upper left', fontsize=9, ncol=2)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('daily_trends_us.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

sector_order = (df_us_sectors.groupby('display_name')['indeed_job_postings_index']
                .median().sort_values(ascending=False).index)

sns.boxplot(data=df_us_sectors, x='display_name', y='indeed_job_postings_index',
            order=sector_order, palette='viridis', ax=ax)

ax.axhline(y=100, color='red', linestyle='--', alpha=0.5, label='Baseline (100)')
ax.set_title('Distribution of Job Postings Index by Sector (US)', fontsize=14)
ax.set_xlabel('Sector')
ax.set_ylabel('Index Value')
ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha='right')
ax.legend()

plt.tight_layout()
plt.savefig('sector_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
coverage = []
for country, df_f in dfs_filtered.items():
    for sector in TARGET_SECTORS:
        n = (df_f['display_name'] == sector).sum()
        coverage.append({'Country': country, 'Sector': sector, 'Days': n})

df_coverage = pd.DataFrame(coverage)
df_cov_pivot = df_coverage.pivot(index='Sector', columns='Country', values='Days')

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(df_cov_pivot, annot=True, fmt='d', cmap='YlGn', ax=ax, linewidths=0.5)
ax.set_title('Data Points per Sector per Country', fontsize=14)
plt.tight_layout()
plt.show()

## 4. Monthly Aggregation

We aggregate daily data to monthly means to reduce noise and align with the project's forecasting horizon (3–6 months). Monthly resolution gives ~75 data points per country per sector.

In [ ]:
monthly_by_country = {}

for country, df_f in dfs_filtered.items():
    df_s = df_f[df_f['display_name'].isin(TARGET_SECTORS)].copy()

    df_pivot = df_s.pivot_table(
        index='date',
        columns='display_name',
        values='indeed_job_postings_index',
        aggfunc='first'
    )

    df_m = df_pivot.resample('ME').mean()
    df_m = df_m.ffill().bfill()

    available = [s for s in TARGET_SECTORS if s in df_m.columns]
    df_m = df_m[available]

    monthly_by_country[country] = df_m
    print(f"{country}: {df_m.shape[0]} months x {df_m.shape[1]} sectors | "
          f"{df_m.index[0].strftime('%b %Y')} to {df_m.index[-1].strftime('%b %Y')} | "
          f"NaN: {df_m.isnull().sum().sum()}")

df_monthly = monthly_by_country['US']
print(f"\nUS monthly data: {df_monthly.shape}")
print(f"Total countries for pooled training: {len(monthly_by_country)}")

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10), sharex=True)
axes = axes.flatten()

for i, sector in enumerate(TARGET_SECTORS):
    ax = axes[i]
    if sector in df_monthly.columns:
        ax.plot(df_monthly.index, df_monthly[sector], color='steelblue', linewidth=2)
    ax.axhline(y=100, color='red', linestyle='--', alpha=0.4)
    ax.set_title(sector, fontsize=11, fontweight='bold')
    ax.set_ylabel('Index')
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis='x', rotation=45)

for j in range(len(TARGET_SECTORS), len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Monthly Job Postings Index per Sector — US', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('monthly_trends_per_sector.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
n_sectors = min(len(TARGET_SECTORS), 8)
n_cols = 4
n_rows = (n_sectors + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 5 * n_rows), sharex=True)
axes = axes.flatten()

country_colors = {'US': 'steelblue', 'AU': 'coral', 'CA': 'green',
                  'DE': 'purple', 'FR': 'orange', 'GB': 'red'}

for i, sector in enumerate(TARGET_SECTORS[:n_sectors]):
    ax = axes[i]
    for country, df_m in monthly_by_country.items():
        if sector in df_m.columns:
            ax.plot(df_m.index, df_m[sector],
                    label=country, color=country_colors.get(country, 'gray'),
                    linewidth=1.5, alpha=0.8)
    ax.axhline(y=100, color='black', linestyle=':', alpha=0.3)
    ax.set_title(sector, fontsize=11, fontweight='bold')
    ax.set_ylabel('Index')
    ax.legend(fontsize=7, ncol=3)
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis='x', rotation=45)

for j in range(n_sectors, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Monthly Trends: All Countries per Sector', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('multi_country_monthly_trends.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

corr = df_monthly.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5, ax=ax,
            vmin=-1, vmax=1)

ax.set_title('Sector Correlation Matrix — US Monthly Index', fontsize=14)
plt.tight_layout()
plt.savefig('sector_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
df_monthly_copy = df_monthly.copy()
df_monthly_copy['month'] = df_monthly_copy.index.month

n_sectors = min(len(TARGET_SECTORS), 8)
n_cols = 4
n_rows = (n_sectors + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 5 * n_rows))
axes = axes.flatten()

for i, sector in enumerate(TARGET_SECTORS[:n_sectors]):
    ax = axes[i]
    if sector in df_monthly_copy.columns:
        monthly_means = df_monthly_copy.groupby('month')[sector].agg(['mean', 'std'])
        ax.bar(monthly_means.index, monthly_means['mean'],
               yerr=monthly_means['std'], capsize=3,
               color='steelblue', alpha=0.7, edgecolor='navy')
    ax.set_title(sector, fontsize=10, fontweight='bold')
    ax.set_xlabel('Month')
    ax.set_ylabel('Avg Index')
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(['J','F','M','A','M','J','J','A','S','O','N','D'])
    ax.grid(True, alpha=0.3, axis='y')

for j in range(n_sectors, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Seasonality Analysis: Average Index by Month — US (with std dev)',
             fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('seasonality_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

df_monthly_copy.drop(columns=['month'], inplace=True)

In [ ]:
print("=" * 70)
print("MULTI-COUNTRY DATASET SUMMARY")
print("=" * 70)

total_months = sum(df_m.shape[0] for df_m in monthly_by_country.values())
print(f"Countries: {len(monthly_by_country)} ({', '.join(monthly_by_country.keys())})")
print(f"Target sectors: {len(TARGET_SECTORS)}")
print(f"Total country-months: {total_months}\n")

for country, df_m in monthly_by_country.items():
    avail = [s for s in TARGET_SECTORS if s in df_m.columns]
    print(f"  {country}: {df_m.shape[0]} months x {len(avail)} sectors")

print(f"\nUS Monthly Statistics:")
summary = df_monthly.describe().T[['mean', 'std', 'min', 'max']]
summary.columns = ['Mean Index', 'Std Dev', 'Min', 'Max']
summary['Range'] = summary['Max'] - summary['Min']
summary = summary.round(2)
print(summary)

## 5. Data Normalization & Cross-Country Sequence Pooling

**MinMaxScaler** maps each country's series to [0, 1] independently. We then create sliding window sequences (12 months) from each country and pool them together for training. Validation uses US data only.

- Training: first 80% of each country's data → sequences pooled
- Validation: last 20% of US data only
- Features: [normalized_index, sin(month), cos(month)]

In [ ]:
WINDOW_SIZE = 12
TRAIN_SPLIT = 0.80

def create_sequences(data, window_size):
    X, y = [], []
    for i in range(len(data) - window_size):
        X.append(data[i : i + window_size])
        y.append(data[i + window_size, 0])
    return np.array(X), np.array(y)

def prepare_single_country(df_m, sector, window_size, train_split):
    series = df_m[sector].values.reshape(-1, 1)

    scaler = MinMaxScaler(feature_range=(0, 1))
    scaled = scaler.fit_transform(series)

    months = df_m.index.month.values
    month_sin = np.sin(2 * np.pi * months / 12).reshape(-1, 1)
    month_cos = np.cos(2 * np.pi * months / 12).reshape(-1, 1)

    features = np.hstack([scaled, month_sin, month_cos])

    split_idx = int(len(features) * train_split)
    train_data = features[:split_idx]
    val_data = features[split_idx - window_size:]

    X_train, y_train = create_sequences(train_data, window_size)
    X_val, y_val = create_sequences(val_data, window_size)

    return X_train, y_train, X_val, y_val, scaler

def prepare_pooled_data(monthly_by_country, sector, window_size, train_split):
    all_X_train, all_y_train = [], []
    us_scaler, us_X_val, us_y_val = None, None, None

    for country, df_m in monthly_by_country.items():
        if sector not in df_m.columns:
            continue

        X_tr, y_tr, X_v, y_v, scaler = prepare_single_country(
            df_m, sector, window_size, train_split
        )

        if len(X_tr) > 0:
            all_X_train.append(X_tr)
            all_y_train.append(y_tr)

        if country == 'US':
            us_scaler = scaler
            us_X_val = X_v
            us_y_val = y_v

    X_train = np.concatenate(all_X_train, axis=0)
    y_train = np.concatenate(all_y_train, axis=0)

    return X_train, y_train, us_X_val, us_y_val, us_scaler

# Quick test
sector_test = TARGET_SECTORS[0]
X_tr, y_tr, X_v, y_v, scaler_test = prepare_pooled_data(
    monthly_by_country, sector_test, WINDOW_SIZE, TRAIN_SPLIT
)

X_tr_us, y_tr_us, _, _, _ = prepare_single_country(
    df_monthly, sector_test, WINDOW_SIZE, TRAIN_SPLIT
)

print(f"Sector: {sector_test}")
print(f"\nSingle-country (US only):")
print(f"  X_train: {X_tr_us.shape}  ({X_tr_us.shape[0]} sequences)")
print(f"\nPooled ({len(monthly_by_country)} countries):")
print(f"  X_train: {X_tr.shape}  ({X_tr.shape[0]} sequences) = {X_tr.shape[0]/max(X_tr_us.shape[0],1):.1f}x more data")
print(f"  X_val:   {X_v.shape}   (US only)")
print(f"\nFeatures per timestep: {X_tr.shape[2]} [index, sin(month), cos(month)]")

In [ ]:
split_month = int(len(df_monthly) * TRAIN_SPLIT)

n_sectors = min(len(TARGET_SECTORS), 8)
n_cols = 4
n_rows = (n_sectors + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 5 * n_rows), sharex=True)
axes = axes.flatten()

for i, sector in enumerate(TARGET_SECTORS[:n_sectors]):
    ax = axes[i]
    if sector in df_monthly.columns:
        ax.plot(df_monthly.index[:split_month], df_monthly[sector].iloc[:split_month],
                color='steelblue', label='Train', linewidth=2)
        ax.plot(df_monthly.index[split_month:], df_monthly[sector].iloc[split_month:],
                color='coral', label='Validation', linewidth=2)
        ax.axvline(x=df_monthly.index[split_month], color='black', linestyle='--', alpha=0.5)
    ax.set_title(sector, fontsize=11, fontweight='bold')
    ax.set_ylabel('Index')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis='x', rotation=45)

for j in range(n_sectors, len(axes)):
    axes[j].set_visible(False)

fig.suptitle(f'Train/Validation Split — US ({int(TRAIN_SPLIT*100)}%/{int((1-TRAIN_SPLIT)*100)}%)',
             fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('train_val_split.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Train: {df_monthly.index[0].strftime('%b %Y')} – {df_monthly.index[split_month-1].strftime('%b %Y')} ({split_month} months)")
print(f"Val:   {df_monthly.index[split_month].strftime('%b %Y')} – {df_monthly.index[-1].strftime('%b %Y')} ({len(df_monthly)-split_month} months)")

## 6. Model Training — LSTM vs GRU vs BiLSTM

We train three recurrent architectures on the same pooled multi-country data and compare their performance:

| Model | Architecture | Key Difference |
|-------|-------------|----------------|
| **LSTM** | 2-layer stacked LSTM (64→32) | Full gating mechanism (input, forget, output gates) |
| **GRU** | 2-layer stacked GRU (64→32) | Simplified gating (reset + update gates), fewer parameters |
| **BiLSTM** | Bidirectional LSTM (64) + LSTM (32) | Processes sequence in both directions |

All models use the same training configuration: Adam optimizer, MSE loss, EarlyStopping with patience=15, batch_size=16.

In [ ]:
def build_lstm_model(input_shape):
    model = Sequential([
        LSTM(64, return_sequences=True, input_shape=input_shape),
        Dropout(0.2),
        LSTM(32, return_sequences=False),
        Dropout(0.2),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    return model

def build_gru_model(input_shape):
    model = Sequential([
        GRU(64, return_sequences=True, input_shape=input_shape),
        Dropout(0.2),
        GRU(32, return_sequences=False),
        Dropout(0.2),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    return model

def build_bilstm_model(input_shape):
    model = Sequential([
        Bidirectional(LSTM(64, return_sequences=True), input_shape=input_shape),
        Dropout(0.2),
        LSTM(32, return_sequences=False),
        Dropout(0.2),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    return model

MODEL_BUILDERS = {
    'LSTM': build_lstm_model,
    'GRU': build_gru_model,
    'BiLSTM': build_bilstm_model,
}

all_results = {name: {} for name in MODEL_BUILDERS}

for sector in TARGET_SECTORS:
    X_train, y_train, X_val, y_val, scaler = prepare_pooled_data(
        monthly_by_country, sector, WINDOW_SIZE, TRAIN_SPLIT
    )

    if X_val is None or len(X_val) == 0:
        print(f"Skipping {sector} — no US validation data")
        continue

    print(f"\n{'='*60}")
    print(f"{sector} | Train: {X_train.shape[0]} seq | Val: {X_val.shape[0]} seq")
    print(f"{'='*60}")

    for model_name, builder in MODEL_BUILDERS.items():
        tf.random.set_seed(SEED)
        np.random.seed(SEED)

        model = builder(input_shape=(X_train.shape[1], X_train.shape[2]))

        early_stop = EarlyStopping(
            monitor='val_loss', patience=15,
            restore_best_weights=True, verbose=0
        )

        history = model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=100, batch_size=16,
            callbacks=[early_stop], verbose=0
        )

        best_epoch = len(history.history['loss']) - early_stop.patience
        print(f"  {model_name:6s} | epochs: {len(history.history['loss']):3d} | "
              f"best ~{max(1, best_epoch):2d} | "
              f"train_loss: {history.history['loss'][-1]:.5f} | "
              f"val_loss: {history.history['val_loss'][-1]:.5f}")

        all_results[model_name][sector] = {
            'model': model,
            'history': history,
            'scaler': scaler,
            'X_train': X_train, 'y_train': y_train,
            'X_val': X_val, 'y_val': y_val,
        }

print(f"\nAll 3 models trained for {len(TARGET_SECTORS)} sectors.")

In [ ]:
n_sectors = len(TARGET_SECTORS)
fig, axes = plt.subplots(n_sectors, 3, figsize=(18, 4 * n_sectors))

model_colors = {'LSTM': 'steelblue', 'GRU': 'green', 'BiLSTM': 'purple'}

for i, sector in enumerate(TARGET_SECTORS):
    for j, model_name in enumerate(MODEL_BUILDERS.keys()):
        ax = axes[i, j]
        if sector in all_results[model_name]:
            h = all_results[model_name][sector]['history'].history
            ax.plot(h['loss'], label='Train', color=model_colors[model_name])
            ax.plot(h['val_loss'], label='Val', color='coral')
        ax.set_title(f"{sector} — {model_name}", fontsize=10, fontweight='bold')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('MSE')
        ax.legend(fontsize=7)
        ax.grid(True, alpha=0.3)

fig.suptitle('Training & Validation Loss: LSTM vs GRU vs BiLSTM', fontsize=16, y=1.01)
plt.tight_layout()
plt.savefig('training_loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Evaluation — LSTM vs GRU vs BiLSTM

We compare the three architectures using RMSE and MAE on the US validation set. This tells us which recurrent model best captures the temporal patterns in the job postings data.

In [ ]:
eval_results = []

for sector in TARGET_SECTORS:
    row = {'Sector': sector}

    for model_name in MODEL_BUILDERS.keys():
        if sector not in all_results[model_name]:
            continue
        r = all_results[model_name][sector]
        scaler = r['scaler']

        y_pred_scaled = r['model'].predict(r['X_val'], verbose=0).flatten()
        y_pred = scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
        y_actual = scaler.inverse_transform(r['y_val'].reshape(-1, 1)).flatten()

        rmse = np.sqrt(mean_squared_error(y_actual, y_pred))
        mae = mean_absolute_error(y_actual, y_pred)

        row[f'{model_name}_RMSE'] = round(rmse, 2)
        row[f'{model_name}_MAE'] = round(mae, 2)

        all_results[model_name][sector]['y_pred'] = y_pred
        all_results[model_name][sector]['y_actual'] = y_actual

    eval_results.append(row)

df_eval = pd.DataFrame(eval_results)

print("=" * 95)
print("MODEL COMPARISON: LSTM vs GRU vs BiLSTM — US Validation Set")
print("=" * 95)
print(df_eval.to_string(index=False))
print()

# Determine best model per sector
for metric in ['RMSE', 'MAE']:
    cols = [f'{m}_{metric}' for m in MODEL_BUILDERS.keys()]
    best = df_eval[cols].idxmin(axis=1).apply(lambda x: x.split('_')[0])
    print(f"Best model by {metric}: {best.value_counts().to_dict()}")

# Overall averages
print("\nAverage metrics:")
for model_name in MODEL_BUILDERS.keys():
    avg_rmse = df_eval[f'{model_name}_RMSE'].mean()
    avg_mae = df_eval[f'{model_name}_MAE'].mean()
    print(f"  {model_name:6s} | RMSE: {avg_rmse:.2f} | MAE: {avg_mae:.2f}")

In [ ]:
split_month = int(len(df_monthly) * TRAIN_SPLIT)
val_dates = df_monthly.index[split_month:]

n_sectors = len(TARGET_SECTORS)
n_cols = 4
n_rows = (n_sectors + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 5 * n_rows))
axes = axes.flatten()

model_styles = {
    'LSTM': ('steelblue', '--'),
    'GRU': ('green', '-.'),
    'BiLSTM': ('purple', ':'),
}

for i, sector in enumerate(TARGET_SECTORS):
    ax = axes[i]
    r = all_results['LSTM'][sector]
    n = len(r['y_actual'])
    dates = val_dates[:n]

    ax.plot(dates, r['y_actual'], label='Actual', color='black', linewidth=2)

    for model_name, (color, ls) in model_styles.items():
        y_pred = all_results[model_name][sector]['y_pred']
        ax.plot(dates, y_pred, label=model_name, color=color, linewidth=1.5, linestyle=ls)

    ax.set_title(sector, fontsize=11, fontweight='bold')
    ax.set_ylabel('Index')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis='x', rotation=45)

for j in range(n_sectors, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('US Validation: Actual vs LSTM vs GRU vs BiLSTM', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

sectors_list = list(TARGET_SECTORS)
x = np.arange(len(sectors_list))
width = 0.25
model_colors = {'LSTM': 'steelblue', 'GRU': 'green', 'BiLSTM': 'purple'}

for idx, metric in enumerate(['RMSE', 'MAE']):
    ax = axes[idx]
    for k, model_name in enumerate(MODEL_BUILDERS.keys()):
        values = df_eval[f'{model_name}_{metric}']
        ax.bar(x + k * width - width, values, width,
               label=model_name, color=model_colors[model_name])
    ax.set_ylabel(f'{metric} (Index Points)')
    ax.set_title(f'{metric} Comparison: LSTM vs GRU vs BiLSTM', fontsize=13)
    ax.set_xticks(x)
    ax.set_xticklabels(sectors_list, rotation=40, ha='right', fontsize=9)
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('metrics_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Future Forecasting (6 Months Ahead)

We use the best-performing model (by RMSE) for each sector to generate autoregressive 6-month forecasts. The last 12 known months are fed in, and each prediction is appended to the window for the next step.

In [ ]:
FORECAST_MONTHS = 6
forecasts = {}

# Pick the best model per sector (lowest RMSE)
best_model_per_sector = {}
for sector in TARGET_SECTORS:
    best_name, best_rmse = None, float('inf')
    for model_name in MODEL_BUILDERS.keys():
        rmse = df_eval.loc[df_eval['Sector'] == sector, f'{model_name}_RMSE'].values[0]
        if rmse < best_rmse:
            best_rmse = rmse
            best_name = model_name
    best_model_per_sector[sector] = best_name

for sector in TARGET_SECTORS:
    model_name = best_model_per_sector[sector]
    r = all_results[model_name][sector]
    scaler = r['scaler']
    model = r['model']

    series = df_monthly[sector].values.reshape(-1, 1)
    scaled_series = scaler.transform(series)

    months = df_monthly.index.month.values
    month_sin = np.sin(2 * np.pi * months / 12).reshape(-1, 1)
    month_cos = np.cos(2 * np.pi * months / 12).reshape(-1, 1)
    features = np.hstack([scaled_series, month_sin, month_cos])

    last_window = features[-WINDOW_SIZE:].copy()
    future_preds = []
    last_month = df_monthly.index[-1].month

    for step in range(FORECAST_MONTHS):
        input_seq = last_window.reshape(1, WINDOW_SIZE, 3)
        pred_scaled = model.predict(input_seq, verbose=0)[0, 0]

        next_month = (last_month % 12) + 1
        next_sin = np.sin(2 * np.pi * next_month / 12)
        next_cos = np.cos(2 * np.pi * next_month / 12)

        pred_original = scaler.inverse_transform([[pred_scaled]])[0, 0]
        future_preds.append(pred_original)

        new_row = np.array([[pred_scaled, next_sin, next_cos]])
        last_window = np.vstack([last_window[1:], new_row])
        last_month = next_month

    last_date = df_monthly.index[-1]
    future_dates = pd.date_range(start=last_date + pd.DateOffset(months=1),
                                  periods=FORECAST_MONTHS, freq='ME')

    forecasts[sector] = {'dates': future_dates, 'values': future_preds}

    print(f"{sector} (using {model_name}):")
    for d, v in zip(future_dates, future_preds):
        direction = "+" if v > series[-1][0] else "-"
        print(f"  {d.strftime('%b %Y')}: {v:.1f} ({direction})")

print(f"\n6-month forecasts generated for all {len(forecasts)} sectors")

In [ ]:
n_sectors = len(TARGET_SECTORS)
n_cols = 4
n_rows = (n_sectors + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 6 * n_rows))
axes = axes.flatten()

for i, sector in enumerate(TARGET_SECTORS):
    ax = axes[i]
    model_used = best_model_per_sector[sector]

    recent = df_monthly[sector].iloc[-24:]
    ax.plot(recent.index, recent.values, color='steelblue', linewidth=2, label='Historical')

    f = forecasts[sector]
    ax.plot(f['dates'], f['values'], color='red', linewidth=2,
            linestyle='--', marker='o', markersize=5, label=f'Forecast ({model_used})')

    ax.plot([recent.index[-1], f['dates'][0]],
            [recent.values[-1], f['values'][0]],
            color='red', linewidth=1, linestyle='--', alpha=0.5)

    for j in range(len(f['values'])):
        uncertainty = (j + 1) * 3
        ax.fill_between([f['dates'][j]], [f['values'][j] - uncertainty],
                        [f['values'][j] + uncertainty], color='red', alpha=0.1)

    ax.axhline(y=100, color='gray', linestyle=':', alpha=0.4)
    ax.set_title(sector, fontsize=11, fontweight='bold')
    ax.set_ylabel('Index')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis='x', rotation=45)

for j in range(n_sectors, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('US Job Market Forecast: Next 6 Months (Best Model per Sector)',
             fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('future_forecast.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print("=" * 80)
print("6-MONTH FORECAST SUMMARY — US Market")
print("=" * 80)

forecast_rows = []
for sector in TARGET_SECTORS:
    current = df_monthly[sector].iloc[-1]
    f = forecasts[sector]
    end_val = f['values'][-1]
    change_pct = ((end_val - current) / current) * 100
    trend = "Growing" if change_pct > 2 else ("Declining" if change_pct < -2 else "Stable")

    forecast_rows.append({
        'Sector': sector,
        'Best Model': best_model_per_sector[sector],
        'Current': round(current, 1),
        'Month_1': round(f['values'][0], 1),
        'Month_3': round(f['values'][2], 1),
        'Month_6': round(f['values'][5], 1),
        'Change_%': round(change_pct, 1),
        'Trend': trend,
    })

df_forecast = pd.DataFrame(forecast_rows)
print(df_forecast.to_string(index=False))

## 9. Interpretation & Limitations

### Architecture Comparison

| Model | Strengths | Weaknesses |
|-------|-----------|-----------|
| **LSTM** | Strong long-term memory via 3 gates | More parameters, slower training |
| **GRU** | Fewer parameters, faster convergence | May miss very long-range dependencies |
| **BiLSTM** | Sees both past and future context in sequence | Double the parameters of unidirectional LSTM |

### Limitations

| Limitation | Impact |
|-----------|--------|
| Different economies pooled | Normalization handles scale, but structural differences remain |
| COVID dominates the training period | Model may overfit to crash-recovery dynamics |
| Small validation set (15 months) | Metrics can be noisy |
| Autoregressive error accumulation | Multi-step forecasts degrade with horizon |
| No exogenous variables | GDP, interest rates, layoff data could improve results |

### Possible Improvements

1. Add attention mechanism to focus on relevant past months
2. Include exogenous features (macro indicators)
3. Use walk-forward validation for more robust metrics
4. Add country embeddings as a learnable feature
5. Ensemble the three models for final predictions

In [ ]:
print("Model Architectures:")
print("=" * 50)
for model_name in MODEL_BUILDERS.keys():
    sample_sector = list(all_results[model_name].keys())[0]
    sample_model = all_results[model_name][sample_sector]['model']
    print(f"\n--- {model_name} ---")
    sample_model.summary()

print(f"\nShared Hyperparameters:")
print(f"  Window size:     {WINDOW_SIZE} months")
print(f"  Features:        3 (index, sin_month, cos_month)")
print(f"  Dropout:         0.2")
print(f"  Optimizer:       Adam (lr=0.001)")
print(f"  Loss:            MSE")
print(f"  Early Stopping:  patience=15")
print(f"  Batch size:      16")
print(f"  Train/Val split: {int(TRAIN_SPLIT*100)}/{int((1-TRAIN_SPLIT)*100)}%")
print(f"  Countries:       {len(monthly_by_country)} ({', '.join(monthly_by_country.keys())})")